# Vision Models

In 2026, the industry standard for the "detect → mask → modify" workflow is built on SAM 2 (Segment Anything Model 2) for masking and Generative Fill (Diffusion Models) for removal or replacement.

pipeline that:
- Detects an object using a prompt (like "the coffee cup").
- Generates a binary mask using SAM 2.
- Removes the object using "Inpainting" (filling the mask with background pixels).




& "C:\Program Files\Git\bin\sh.exe" ./download_ckpts.sh

In [4]:
import os
import hydra
from hydra.core.global_hydra import GlobalHydra

# 1. Clear any stuck Hydra instances from previous errors
GlobalHydra.instance().clear()

# 2. Define the path to where you cloned SAM 2
# Note: Adjust this path to the EXACT location of the 'sam2' folder 
# that contains the 'configs' directory.
base_path = os.path.abspath("../../sam2") 

# 3. Tell Hydra: "Hey, look here for the recipes!"
hydra.initialize_config_dir(config_dir=os.path.join(base_path, "sam2", "configs"), version_base="1.2")

print(f"✅ Hydra is now looking for configs in: {os.path.join(base_path, 'sam2', 'configs')}")

✅ Hydra is now looking for configs in: c:\personal\genai\sam2\sam2\configs


In [6]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Update these paths to point to your new 'sam2' folder
# 'configs/sam2.1/sam2.1_hiera_l.yaml' is the standard path within the repo
checkpoint = "../../sam2/checkpoints/sam2.1_hiera_large.pt"
# Since we told Hydra to look in the 'configs' folder already, 
# we just need the sub-path.
model_cfg = "sam2.1/sam2.1_hiera_l.yaml"

# This should now run without the MissingConfigException!
sam2_model = build_sam2(model_cfg, checkpoint, device="cuda" if torch.cuda.is_available() else "cpu")
predictor = SAM2ImagePredictor(sam2_model)

print("🎉 Success! The vision agent is alive.")

🎉 Success! The vision agent is alive.


In [10]:
import torch
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from diffusers import AutoPipelineForInpainting
import hydra

# 1. SETUP: Load the Vision Models
device = "cuda" if torch.cuda.is_available() else "cpu"

# # 1. Clear any existing Hydra instance
# hydra.core.global_hydra.GlobalHydra.instance().clear()

# # 2. Initialize with the path to your sam2_configs folder
# # Replace 'path/to/sam2' with the actual path to your cloned repository
# hydra.initialize_config_module('sam2_configs', version_base='1.2')

# # 3. Now build the model using just the filename
# sam2_model = build_sam2("sam2_hiera_large.yaml", "sam2_hiera_large.pt", device=device)
# predictor = SAM2ImagePredictor(sam2_model)

# Load the Inpainter (for removing/replacing)
pipe = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1", 
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to(device)

def smart_remove(image_path, box_prompt):
    """
    box_prompt: [x_min, y_min, x_max, y_max] 
    (Usually provided by a detection model like YOLO or Grounding DINO)
    """
    raw_image = Image.open(image_path).convert("RGB")
    predictor.set_image(raw_image)

    # 2. MASKING: Generate the mask based on the detection box
    masks, scores, _ = predictor.predict(
        box=box_prompt,
        multimask_output=False
    )
    mask_image = Image.fromarray(masks[0]) # This is your 'black and white' mask

    # 3. REMOVAL: Use the mask to tell the Diffusion model what to 'fix'
    # We provide an empty prompt ("") to tell it to just fill with background
    output = pipe(
        prompt="clean background, high quality",
        image=raw_image,
        mask_image=mask_image
    ).images[0]
    
    return output, mask_image

# EXECUTION
# Let's say we have an image and we want to remove an object at these coordinates
final_img, mask_img = smart_remove("./data/pic.jpg", [40, 100, 300, 400])
final_img.save("./data/removed_object.png")

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  6.77it/s]
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fai

: 

In [9]:
import os

# 1. Print where the computer is currently 'standing'
print(f"📍 Current Working Directory: {os.getcwd()}")

# 2. Check if the folder exists
print(f"📁 Does 'data' folder exist? {os.path.exists('./data')}")

# 3. Check if the specific file exists
file_to_check = "./data/pic.jpg"
print(f"📄 Does 'pic.jpg' exist at {file_to_check}? {os.path.exists(file_to_check)}")

# 4. List all files in the data folder (to see if there's a typo)
if os.path.exists('./data'):
    print(f"🔎 Files found in 'data/': {os.listdir('./data')}")

📍 Current Working Directory: c:\personal\genai\gen-ai\new
📁 Does 'data' folder exist? False
📄 Does 'pic.jpg' exist at ./data/pic.jpg? False
